In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import pandas as pd

In [2]:
data_path = './data/preprocessed/'
# Load preprocessed data
train_dataset = torch.load(data_path+'train_dataset.pth', weights_only=False)
test_dataset = torch.load(data_path+'test_dataset.pth', weights_only=False)
# label_encoder = joblib.load(data_path+'label_encoder.pkl')

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [3]:
# Get all samples from train_dataset
X_train_all, y_train_all = train_dataset[:]  # or iterate through

# Assuming y is one-hot encoded: [1,0] for Success, [0,1] for Anomaly
# Find indices of success cases
success_indices = (y_train_all.argmax(dim=1) == 1).nonzero().squeeze()

# Extract only success cases
X_success = X_train_all[success_indices]
y_success = y_train_all[success_indices]

In [4]:
import torch.nn.functional as F

# 1. Normalize your embeddings first
X_success_normalized = [F.normalize(seq, dim=-1) for seq in X_success]

In [5]:
class NextEventDataset(torch.utils.data.Dataset):
    def __init__(self, sequences, max_len=50):
        """
        sequences: list of tensors, each [seq_len, embed_dim]
        Handles LEFT-padded sequences (zeros at start)
        """
        self.sequences = sequences
        self.max_len = max_len
        self.pairs = []
        
        print(f"Processing {len(sequences)} sequences...")
        
        for seq_idx, seq in enumerate(sequences):
            # Find where actual data starts (first non-zero row)
            # Method 1: Check if any element is non-zero
            non_zero_mask = (seq != 0).any(dim=1)  # [seq_len]
            
            if not non_zero_mask.any():
                print(f"⚠️ Sequence {seq_idx} is all zeros, skipping")
                continue
                
            # Get actual start and end of data
            data_start = non_zero_mask.nonzero()[0, 0].item() if non_zero_mask.any() else 0
            data_end = non_zero_mask.nonzero()[-1, 0].item() + 1 if non_zero_mask.any() else len(seq)
            
            actual_data = seq[data_start:data_end]
            actual_len = len(actual_data)
            
            if actual_len < 2:
                continue  # Need at least 2 events for prediction
                
            # Create (context, target) pairs from ACTUAL data only
            for i in range(1, actual_len):
                context = actual_data[:i]  # All previous actual events
                target = actual_data[i]     # Next actual event
                
                # Truncate if too long
                if len(context) > max_len:
                    context = context[-max_len:]
                
                self.pairs.append((context, target))
                
            if seq_idx < 3:  # Debug first few
                print(f"Seq {seq_idx}: padded_len={len(seq)}, actual_len={actual_len}, pairs={actual_len-1}")
        
        print(f"Created {len(self.pairs)} training pairs")
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        return self.pairs[idx]

In [6]:
def left_pad_collate(batch):
    """Left-pad sequences (zeros at start)"""
    contexts, targets = zip(*batch)
    
    # Get max length in this batch
    max_len = max(len(ctx) for ctx in contexts)
    embed_dim = contexts[0].shape[-1]
    batch_size = len(contexts)
    
    # Create left-padded batch
    padded = torch.zeros(batch_size, max_len, embed_dim)
    padding_mask = torch.ones(batch_size, max_len, dtype=torch.bool)
    
    for i, ctx in enumerate(contexts):
        ctx_len = len(ctx)
        start_idx = max_len - ctx_len  # Left padding
        padded[i, start_idx:] = ctx
        padding_mask[i, start_idx:] = False
    
    return padded, torch.stack(targets), padding_mask

In [7]:
dataset = NextEventDataset(X_success_normalized)
next_event_loader = DataLoader(
    dataset, 
    batch_size=64, 
    shuffle=True,
    collate_fn=left_pad_collate
)


Processing 11399 sequences...
Seq 0: padded_len=50, actual_len=30, pairs=29
Seq 1: padded_len=50, actual_len=36, pairs=35
Seq 2: padded_len=50, actual_len=22, pairs=21
Created 290772 training pairs


In [8]:
import torch
import torch.nn as nn

class NextEventTransformer(nn.Module):
    def __init__(self, embed_dim=384, num_heads=8, num_layers=6,
                 max_seq_len=50, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len

        # learned positional embeddings
        self.pos_emb = nn.Parameter(
            torch.randn(1, max_seq_len, embed_dim) * 0.02
        )

        # encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4 * embed_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            layer_norm_eps=1e-6
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # prediction head
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 2 * embed_dim),
            nn.GELU(),
            nn.Linear(2 * embed_dim, embed_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for p in self.encoder.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, padding_mask=None):
        # x: (B, T, D)
        
        # Use provided mask or detect zeros
        if padding_mask is None:
            padding_mask = (x.sum(dim=-1) == 0)
        
        # Add positional embeddings
        x = x + self.pos_emb[:, :x.size(1)]
        x = x * (self.embed_dim ** 0.5)
        
        # Transformer
        enc = self.encoder(x, src_key_padding_mask=padding_mask)
        
        # Get last non-padding token
        lengths = (~padding_mask).sum(dim=1)
        idx = torch.clamp(lengths - 1, min=0)
        
        # Gather last hidden states
        b = x.size(0)
        last_h = enc[torch.arange(b), idx]
        
        # Predict next embedding
        return self.head(last_h)


In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = NextEventTransformer(embed_dim=384).to(device)

In [ ]:
# print("="*50)
# print("QUICK DIAGNOSTICS")
# print("="*50)

# # Get one batch
# batch_X, batch_y, mask = next(iter(next_event_loader))
# print(f"Batch X shape: {batch_X.shape}")
# print(f"Batch y shape: {batch_y.shape}")
# print(f"Mask shape: {mask.shape}")

# # Check padding percentage
# padding_pct = mask.float().mean()*100
# print(f"\nPadding: {padding_pct:.1f}%")

# # Check non-padding values (FIXED)
# if mask.shape[1] == batch_X.shape[1]:  # Mask matches sequence length
#     # Expand mask to match X dimensions
#     mask_expanded = mask.unsqueeze(-1).expand_as(batch_X)
#     non_pad = batch_X[~mask_expanded].view(-1, batch_X.shape[-1])
    
#     print(f"\nNon-padding values:")
#     print(f"  Count: {len(non_pad)} vectors")
#     if len(non_pad) > 0:
#         print(f"  Min: {non_pad.min():.4f}, Max: {non_pad.max():.4f}")
#         print(f"  Norms: {torch.norm(non_pad, dim=-1).mean():.4f} (should be ~1.0)")
#     else:
#         print("  No non-padding values found!")
# else:
#     print(f"\n⚠️ Mask shape {mask.shape} doesn't match X shape {batch_X.shape}")

# # Check targets
# print(f"\nTargets:")
# print(f"  Norms: {torch.norm(batch_y, dim=-1).mean():.4f} (should be ~1.0)")

# with torch.no_grad():
#     out = model(batch_X.to(device), mask.to(device))
#     out_norm = F.normalize(out, dim=-1)
#     target_norm = F.normalize(batch_y.to(device), dim=-1)
#     cos_sim = (out_norm * target_norm).sum(-1).mean()
#     print(f"\nModel test:")
#     print(f"  Output shape: {out.shape}")
#     print(f"  Initial cosine sim: {cos_sim:.4f}")
#     print(f"  Initial loss: {1-cos_sim:.4f}")

In [ ]:
import torch
import torch.nn.functional as F
import os
import math
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NextEventTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

warmup_steps = 500
base_lr = 5e-5
total_steps = len(next_event_loader) * 5
global_step = 0

# Save directory
save_dir = "models/transformer/checkpoints"
os.makedirs(save_dir, exist_ok=True)

# Track best model
best_loss = float('inf')
best_model_state = None
patience = 1  # Stop if loss doesn't improve for 2 epochs
patience_counter = 0

# Training loop
for epoch in range(5):
    model.train()
    pbar = tqdm(next_event_loader, desc=f"Epoch {epoch+1}", ncols=100)
    
    epoch_loss = 0
    num_batches = 0

    for batch_X, batch_y, padding_mask in pbar:
        global_step += 1

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        padding_mask = padding_mask.to(device)

        # ---- LR schedule ----
        if global_step < warmup_steps:
            lr = base_lr * (global_step / warmup_steps)
        else:
            progress = (global_step - warmup_steps) / (total_steps - warmup_steps)
            progress = min(progress, 1.0)
            lr = base_lr * 0.5 * (1 + math.cos(math.pi * progress))

        for g in optimizer.param_groups:
            g["lr"] = lr

        # ---- forward ----
        pred = model(batch_X, padding_mask)
        pred_n = F.normalize(pred, dim=-1)
        tgt_n = F.normalize(batch_y, dim=-1)

        loss = 1 - (pred_n * tgt_n).sum(-1).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": lr})

    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    print(f"\nEpoch {epoch+1} completed. Average loss: {avg_epoch_loss:.4f}")
    
    # Check if this is the best model
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        best_model_state = model.state_dict().copy()  # Save best weights
        patience_counter = 0  # Reset patience
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, os.path.join(save_dir, "best_model.pth"))
        print(f"✅ New best model! Loss: {best_loss:.4f} (Cosine similarity: {1-best_loss:.4f})")
    else:
        patience_counter += 1
        print(f"⚠️  Loss didn't improve. Patience: {patience_counter}/{patience}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\n🚨 Early stopping triggered! No improvement for {patience} epochs.")
        print(f"Best loss: {best_loss:.4f} at epoch {epoch+1-patience}")
        break

# Load the best model weights back
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nLoaded best model with loss: {best_loss:.4f}")

# Save final model (with best weights)
torch.save({
    'epoch': epoch,
    'global_step': global_step,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': best_loss,
}, os.path.join(save_dir, "final_model.pth"))

print(f"\n🏆 Training complete! Best cosine similarity: {1-best_loss:.4f}")
print(f"Models saved to: {save_dir}/")

In [ ]:
checkpoint = torch.load("models/transformer/checkpoints/best_model.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [27]:
# Test on the SAME type of data you trained on
print("Testing on next_event_loader data (embeddings)...")

# Get a batch from next_event_loader
test_batch_X, test_batch_y, test_mask = next(iter(next_event_loader))
test_batch_X, test_batch_y, test_mask = test_batch_X.to(device), test_batch_y.to(device), test_mask.to(device)

# Calculate loss the SAME way as training
with torch.no_grad():
    pred = model(test_batch_X, test_mask)
    pred_n = F.normalize(pred, dim=-1)
    tgt_n = F.normalize(test_batch_y, dim=-1)
    
    test_loss = 1 - (pred_n * tgt_n).sum(-1).mean()
    test_similarity = 1 - test_loss.item()

print(f"Test loss: {test_loss.item():.6f}")
print(f"Test similarity: {test_similarity:.6f}")
print(f"Training loss was: {checkpoint['loss']:.6f}")
print(f"Should be similar!")

Testing on next_event_loader data (embeddings)...
Test loss: 0.066336
Test similarity: 0.933664
Training loss was: 0.063404
Should be similar!


In [29]:
def get_prediction_similarity(model, context, target):
    """Calculate cosine similarity between prediction and target"""
    device = next(model.parameters()).device
    
    # Prepare input
    if len(context) > model.max_seq_len:
        context = context[-model.max_seq_len:]
    
    if len(context) < model.max_seq_len:
        padded = torch.zeros((model.max_seq_len, context.shape[-1]), device=device)
        start_idx = model.max_seq_len - len(context)
        padded[start_idx:] = context.to(device)
        mask = torch.ones(model.max_seq_len, dtype=torch.bool, device=device)
        mask[start_idx:] = False
    else:
        padded = context.to(device)
        mask = torch.zeros(model.max_seq_len, dtype=torch.bool, device=device)
    
    # Predict
    with torch.no_grad():
        pred = model(padded.unsqueeze(0), mask.unsqueeze(0))  # [1, 384]
        pred_n = F.normalize(pred.squeeze(0), dim=-1)  # [384]
        target_n = F.normalize(target.to(device), dim=-1)  # [384]
        
        cos_sim = (pred_n * target_n).sum().item()
    
    return cos_sim

# # Try predicting on your ORIGINAL training data
# print("\nTesting on ORIGINAL X_success (training data):")
# if 'X_success' in locals() and len(X_success) > 0:
#     train_seq = X_success[0]

#     # Use same prediction code
#     context = train_seq[:20]  # First 20 events
#     target = train_seq[20]    # 21st event
    
#     # Calculate similarity
#     sim = get_prediction_similarity(model, context, target)
#     print(f"Training sequence similarity: {sim:.4f}")
#     print(f"Should be ~0.93 like your training loss suggests!")
# else:
#     print("X_success not found")

In [ ]:
def test_anomaly_detection(model, test_dataset, threshold=0.7):
    """Test if low similarity catches Fail sequences"""
    
    success_similarities = []
    fail_similarities = []
    
    for i in range(len(test_dataset)):
        seq, label = test_dataset[i]
        
        # Skip if too short
        if len(seq) < 15:
            continue
        
        # Clean sequence
        non_zero = (seq != 0).any(dim=1)
        if not non_zero.any():
            continue
        
        start_idx = non_zero.nonzero()[0, 0].item()
        clean_seq = seq[start_idx:]
        
        # Calculate average similarity for this sequence
        seq_similarities = []
        for pos in range(5, len(clean_seq) - 1):
            context = clean_seq[:pos+1]
            target = clean_seq[pos+1]
            sim = get_prediction_similarity(model, context, target)
            seq_similarities.append(sim)
        
        if seq_similarities:
            avg_sim = np.mean(seq_similarities)
            
            # Classify based on label
            if label[1] == 1:  # Success
                success_similarities.append(avg_sim)
            else:  # Fail
                fail_similarities.append(avg_sim)
    
    print(f"\nSuccess sequences: {len(success_similarities)}")
    print(f"Fail sequences: {len(fail_similarities)}")
    
    if success_similarities and fail_similarities:
        print(f"\nSuccess avg similarity: {np.mean(success_similarities):.4f}")
        print(f"Fail avg similarity: {np.mean(fail_similarities):.4f}")
        print(f"Difference: {np.mean(success_similarities) - np.mean(fail_similarities):.4f}")
        
        # Check threshold
        success_below = np.sum(np.array(success_similarities) < threshold) / len(success_similarities)
        fail_below = np.sum(np.array(fail_similarities) < threshold) / len(fail_similarities)
        
        print(f"\nWith threshold {threshold}:")
        print(f"  Success flagged as anomaly: {success_below*100:.1f}% (false alarms)")
        print(f"  Fail flagged as anomaly: {fail_below*100:.1f}% (correct detections)")
    
    return success_similarities, fail_similarities

# Run the real test
success_sims, fail_sims = test_anomaly_detection(model, test_dataset, threshold=0.83)


Success sequences: 2857
Fail sequences: 814

Success avg similarity: 0.8887
Fail avg similarity: 0.8325
Difference: 0.0562

With threshold 0.83:
  Success flagged as anomaly: 5.8% (false alarms)
  Fail flagged as anomaly: 40.7% (correct detections)


In [39]:
# Try different thresholds
thresholds = [0.84, 0.85, 0.86, 0.87, 0.88, 0.89]

print("\nTesting different thresholds:")
for thresh in thresholds:
    if success_sims and fail_sims:
        success_flagged = np.sum(np.array(success_sims) < thresh) / len(success_sims) * 100
        fail_flagged = np.sum(np.array(fail_sims) < thresh) / len(fail_sims) * 100
        
        print(f"Threshold {thresh}:")
        print(f"  Success flagged: {success_flagged:.1f}%")
        print(f"  Fail flagged: {fail_flagged:.1f}%")


Testing different thresholds:
Threshold 0.84:
  Success flagged: 8.6%
  Fail flagged: 47.5%
Threshold 0.85:
  Success flagged: 13.3%
  Fail flagged: 54.8%
Threshold 0.86:
  Success flagged: 19.3%
  Fail flagged: 62.3%
Threshold 0.87:
  Success flagged: 26.8%
  Fail flagged: 70.1%
Threshold 0.88:
  Success flagged: 35.8%
  Fail flagged: 78.4%
Threshold 0.89:
  Success flagged: 46.8%
  Fail flagged: 85.6%


In [40]:
# Calculate final metrics for threshold 0.86 from your data
success_count = 317  # From your test
fail_count = 83      # From your test
TPR = 62.3           # True Positive Rate from threshold 0.86 (62.3% Fail caught)
FPR = 19.3           # False Positive Rate from threshold 0.86 (19.3% Success false alarms)

# Calculate accuracy
true_positives = fail_count * (TPR / 100)
true_negatives = success_count * ((100 - FPR) / 100)
total = success_count + fail_count
accuracy = (true_positives + true_negatives) / total * 100

print(f"=== ANOMALY DETECTION PERFORMANCE ===")
print(f"Threshold: 0.86")
print(f"Dataset: {success_count} Success, {fail_count} Fail sequences")
print(f"True Positive Rate (catch Fails): {TPR:.1f}%")
print(f"False Positive Rate (false alarms): {FPR:.1f}%")
print(f"Overall accuracy: {accuracy:.1f}%")
print(f"Precision: {true_positives/(true_positives + success_count*(FPR/100))*100:.1f}%")

=== ANOMALY DETECTION PERFORMANCE ===
Threshold: 0.86
Dataset: 317 Success, 83 Fail sequences
True Positive Rate (catch Fails): 62.3%
False Positive Rate (false alarms): 19.3%
Overall accuracy: 76.9%
Precision: 45.8%
